# Combine the L-curves of interseismic locking and coseismic slip inversion




In [ ]:
# Load libraries
import numpy as np
import pygmt
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import rc
from io import StringIO
import utils_plot as utp

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['OPENBLAS_NUM_THREADS'] = '1'
os.environ['MKL_NUM_THREADS'] = '1'
os.environ['VECLIB_MAXIMUM_THREADS'] = '1'
os.environ['NUMEXPR_NUM_THREADS'] = '1'

## Interseismic inversion parameters

In [ ]:
# work dir
datadir = "/home/staff/chao/SSEinv/Nicoya/"

# Define the inversion results path
resultpath = "rst_locking/"

# Import GNSS data, originally from Feng et al. 2012, but no volcano sites, both trench-parallel and normal components
obs_disp_name = "data/CKfig6_data_final.csv"   # the EXACT data file for figure 6 in Kyriakopoulos et al. (2016)

# the processed data has the unit of m/yr that was converted from mm/yr
df = pd.read_csv(datadir + obs_disp_name, sep=",", skiprows=1, \
                   names=['lon', 'lat', 'vx_Car', 'vy_Car', 'vz_Car', \
                          'vx_std_Car', 'vy_std_Car', 'vz_std_Car'])
# Decide the weights of the horizontal, vertical components
f_h, f_v = 1, 1
print( "Data weight horizontal / vertical: %.2f / %.2f" %(f_h, f_v) )
w_h, w_v = int(1/f_h), int(1/f_v)

# Define the reference point (center of the projection)
lon0, lat0 = -84, 7     # from Christos's email

# Mesh selection (uneven-top mesh; legacy even-top branch removed)
meshnameCK = "nicoyaCKden_une_all"   # uneven top boundary, all subduction interface

def mu_expression(m):
    mu = 20*(2. + np.tanh(m))
    return mu

# background shear modulus
mu_b = 0   # 40 GPa
mu_background = mu_expression(mu_b)

# shear modulus for the lower (subducting) plate
mu_l = 0.9730 # ~55 GPa
mu_lower = mu_expression(mu_l)

# shear modulus for the upper (overriding) plate
mu_u = -0.9730  # ~25 GPa
mu_upper = mu_expression(mu_u)

# Model identifiers
homo_str = f"_mul{round(mu_expression(mu_b))}u{round(mu_expression(mu_b))}"
sw_str   = f"_mul{round(mu_expression(mu_l))}u{round(mu_expression(mu_u))}"
het3d_str_ori = f"_DeShon3D_ref_{round(1.0)}_hull"
contrast_factor = 2.5  # adopted since 03/05/2026
het3d_str   = f"_DeShon3D_ref1D_{round(contrast_factor)}_hull"
het3d_str_4 = f"_DeShon3D_ref1D_{round(4.0)}_hull"

CG_mu_deg = 2   # 1 for hom or SW, 2 for 3D
if CG_mu_deg == 2:
    het3d_str_ori = het3d_str_ori + f"_CG{CG_mu_deg}"
    het3d_str     = het3d_str     + f"_CG{CG_mu_deg}"
    het3d_str_4   = het3d_str_4   + f"_CG{CG_mu_deg}"

rho_s = 2.5e8   # ~15 km, close to the maximum resolution

# Helper: read an L-curve summary (4-column format with gamma + rho_s) and
# sort rows by gamma ascending so the order matches the actual saved L-curve
# regardless of how gammas_s was written by the inversion script.
def _load_lc(mu_str):
    fname = f"Lcurvelockboth_rs{rho_s:.0e}_{meshnameCK}_{mu_str}.txt"
    print(fname)
    df = pd.read_csv(datadir + resultpath + fname, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

# Helper: row index whose gamma is closest to a requested preferred value
def _idx_for_gamma(df, gamma_pref):
    return int((df['gamma'] - gamma_pref).abs().idxmin())

misfitsCK_hom    = _load_lc(homo_str)
misfitsCK_sw     = _load_lc(sw_str)
misfitsCK_3d_ori = _load_lc(het3d_str_ori)
misfitsCK_3d     = _load_lc(het3d_str)
misfitsCK_3d4    = _load_lc(het3d_str_4)

# Preferred gamma per model (just one value here; closest-row is picked from the file)
gamma_CK_hom_prefer    = 3e3
gamma_CK_sw_prefer     = 3e3
gamma_CK_3d_ori_prefer = 3e3
gamma_CK_3d_prefer     = 3e3
gamma_CK_3d4_prefer    = 3e3

idxCK_hom    = _idx_for_gamma(misfitsCK_hom,    gamma_CK_hom_prefer)
idxCK_sw     = _idx_for_gamma(misfitsCK_sw,     gamma_CK_sw_prefer)
idxCK_3d_ori = _idx_for_gamma(misfitsCK_3d_ori, gamma_CK_3d_ori_prefer)
idxCK_3d     = _idx_for_gamma(misfitsCK_3d,     gamma_CK_3d_prefer)
idxCK_3d4    = _idx_for_gamma(misfitsCK_3d4,    gamma_CK_3d4_prefer)

print(idxCK_hom,    misfitsCK_hom['gamma'].iloc[idxCK_hom])
print(idxCK_sw,     misfitsCK_sw['gamma'].iloc[idxCK_sw])
print(idxCK_3d_ori, misfitsCK_3d_ori['gamma'].iloc[idxCK_3d_ori])
print(idxCK_3d,     misfitsCK_3d['gamma'].iloc[idxCK_3d])
print(idxCK_3d4,    misfitsCK_3d4['gamma'].iloc[idxCK_3d4])


## Coseismic inversion parameters

In [ ]:
# Define the inversion results path
resultpath_co = "rst_coseis/"

# coseismic data file
obs_disp_name_co = "data/Protti_etal_2014_tableS1.csv"

df_co = pd.read_csv(datadir + obs_disp_name_co, sep=",", skiprows=1, \
                 names=['site', 'lon', 'lat', 'elv', 'ux', 'uy', 'uz', \
                        'ux_std', 'uy_std', 'uz_std', 'date0', 'date1', 'duration'])

rho_s_co = 2e7   # ~4.5 km, close to the maximum resolution

# Same helper pattern as the interseismic block, just with the coseismic prefix / rho.
def _load_lc_co(mu_str):
    fname = f"Lcurvecoseis_rs{rho_s_co:.0e}_{meshnameCK}_{mu_str}.txt"
    print(fname)
    df = pd.read_csv(datadir + resultpath_co + fname, sep=r'\s+',
                     names=['d_mis', 'm_mis', 'gamma', 'rho_s'])
    return df.sort_values('gamma').reset_index(drop=True)

misfitsCK_hom_co    = _load_lc_co(homo_str)
misfitsCK_sw_co     = _load_lc_co(sw_str)
misfitsCK_3d_ori_co = _load_lc_co(het3d_str_ori)
misfitsCK_3d_co     = _load_lc_co(het3d_str)
misfitsCK_3d4_co    = _load_lc_co(het3d_str_4)

gamma_CK_hom_prefer_co    = 6e2
gamma_CK_sw_prefer_co     = 6e2
gamma_CK_3d_ori_prefer_co = 6e2
gamma_CK_3d_prefer_co     = 6e2
gamma_CK_3d4_prefer_co    = 6e2

idxCK_hom_co    = _idx_for_gamma(misfitsCK_hom_co,    gamma_CK_hom_prefer_co)
idxCK_sw_co     = _idx_for_gamma(misfitsCK_sw_co,     gamma_CK_sw_prefer_co)
idxCK_3d_ori_co = _idx_for_gamma(misfitsCK_3d_ori_co, gamma_CK_3d_ori_prefer_co)
idxCK_3d_co     = _idx_for_gamma(misfitsCK_3d_co,     gamma_CK_3d_prefer_co)
idxCK_3d4_co    = _idx_for_gamma(misfitsCK_3d4_co,    gamma_CK_3d4_prefer_co)

print(idxCK_hom_co,    misfitsCK_hom_co['gamma'].iloc[idxCK_hom_co])
print(idxCK_sw_co,     misfitsCK_sw_co['gamma'].iloc[idxCK_sw_co])
print(idxCK_3d_ori_co, misfitsCK_3d_ori_co['gamma'].iloc[idxCK_3d_ori_co])
print(idxCK_3d_co,     misfitsCK_3d_co['gamma'].iloc[idxCK_3d_co])
print(idxCK_3d4_co,    misfitsCK_3d4_co['gamma'].iloc[idxCK_3d4_co])


In [ ]:
# Plot L-curve (3D variants only — original / amplified x2.5 / amplified x4)
# Gamma labels are read directly from the L-curve dataframes (sorted by gamma) so that
# the plotted text matches whatever gamma was actually saved by the inversion script.
fig, axes = plt.subplots(1, 2, figsize=(8,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

ax = axes[0]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_3d_ori['m_mis'], misfitsCK_3d_ori['d_mis'],
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Original 3D")
ax.plot(misfitsCK_3d_ori.iloc[idxCK_3d_ori]['m_mis'], misfitsCK_3d_ori.iloc[idxCK_3d_ori]['d_mis'],
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=6,
        label=r"Original 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori['gamma'].iloc[idxCK_3d_ori]))

ax.plot(misfitsCK_3d['m_mis'], misfitsCK_3d['d_mis'],
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x2.5")
ax.plot(misfitsCK_3d.iloc[idxCK_3d]['m_mis'], misfitsCK_3d.iloc[idxCK_3d]['d_mis'],
        'D', color='darkblue', markerfacecolor='darkblue', markersize=6,
        label=r"Amplified 3D x2.5 preferred $\gamma$={:.1e}".format(misfitsCK_3d['gamma'].iloc[idxCK_3d]))

ax.plot(misfitsCK_3d4['m_mis'], misfitsCK_3d4['d_mis'],
        'D-', color='darkgreen', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x4")
ax.plot(misfitsCK_3d4.iloc[idxCK_3d4]['m_mis'], misfitsCK_3d4.iloc[idxCK_3d4]['d_mis'],
        'D', color='darkgreen', markerfacecolor='darkgreen', markersize=6,
        label=r"Amplified 3D x4 preferred $\gamma$={:.1e}".format(misfitsCK_3d4['gamma'].iloc[idxCK_3d4]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


ax = axes[1]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

ax.plot(misfitsCK_3d_ori_co['m_mis'], misfitsCK_3d_ori_co['d_mis'],
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Original 3D")
ax.plot(misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['m_mis'], misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['d_mis'],
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=6,
        label=r"Original 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori_co['gamma'].iloc[idxCK_3d_ori_co]))

ax.plot(misfitsCK_3d_co['m_mis'], misfitsCK_3d_co['d_mis'],
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x2.5")
ax.plot(misfitsCK_3d_co.iloc[idxCK_3d_co]['m_mis'], misfitsCK_3d_co.iloc[idxCK_3d_co]['d_mis'],
        'D', color='darkblue', markerfacecolor='darkblue', markersize=6,
        label=r"Amplified 3D x2.5 preferred $\gamma$={:.1e}".format(misfitsCK_3d_co['gamma'].iloc[idxCK_3d_co]))

ax.plot(misfitsCK_3d4_co['m_mis'], misfitsCK_3d4_co['d_mis'],
        'D-', color='darkgreen', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x4")
ax.plot(misfitsCK_3d4_co.iloc[idxCK_3d4_co]['m_mis'], misfitsCK_3d4_co.iloc[idxCK_3d4_co]['d_mis'],
        'D', color='darkgreen', markerfacecolor='darkgreen', markersize=6,
        label=r"Amplified 3D x4 preferred $\gamma$={:.1e}".format(misfitsCK_3d4_co['gamma'].iloc[idxCK_3d4_co]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


In [ ]:
# Plot L-curve (HOM, SW, Original 3D, Amplified 3D x2.5)
# Uses the uneven-mesh format (all rows kept) and reads gamma directly from the dataframes.
fig, axes = plt.subplots(1, 2, figsize=(8,4), dpi=300)

panel_labels = ["(a)", "(b)", "(c)", "(d)"]

ax = axes[0]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_ylabel(r"$||\mathbf{Gm} - \mathbf{d}||_{L_2}$", fontsize=9)
ax.set_title("Interseismic locking ratio inversion", fontsize=9)
ax.tick_params(labelsize=8)

color_L = ['silver', 'firebrick', 'white']

ax.plot(misfitsCK_hom['m_mis'], misfitsCK_hom['d_mis'],
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
ax.plot(misfitsCK_hom.iloc[idxCK_hom]['m_mis'], misfitsCK_hom.iloc[idxCK_hom]['d_mis'],
        'o', color='k', markerfacecolor='k', markersize=5,
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom['gamma'].iloc[idxCK_hom]))

ax.plot(misfitsCK_sw['m_mis'], misfitsCK_sw['d_mis'],
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
ax.plot(misfitsCK_sw.iloc[idxCK_sw]['m_mis'], misfitsCK_sw.iloc[idxCK_sw]['d_mis'],
        's', color='red', markerfacecolor='red', markersize=5,
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw['gamma'].iloc[idxCK_sw]))

ax.plot(misfitsCK_3d_ori['m_mis'], misfitsCK_3d_ori['d_mis'],
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Original 3D")
ax.plot(misfitsCK_3d_ori.iloc[idxCK_3d_ori]['m_mis'], misfitsCK_3d_ori.iloc[idxCK_3d_ori]['d_mis'],
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5,
        label=r"Original 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori['gamma'].iloc[idxCK_3d_ori]))

ax.plot(misfitsCK_3d['m_mis'], misfitsCK_3d['d_mis'],
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x2.5")
ax.plot(misfitsCK_3d.iloc[idxCK_3d]['m_mis'], misfitsCK_3d.iloc[idxCK_3d]['d_mis'],
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5,
        label=r"Amplified 3D x2.5 preferred $\gamma$={:.1e}".format(misfitsCK_3d['gamma'].iloc[idxCK_3d]))

ax.text(0.02, 0.02, panel_labels[0], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)


ax = axes[1]
ax.set_xlabel(r"$||\nabla \mathbf{m}||_{L_2}$", fontsize=9)
ax.set_title("Coseismic slip inversion", fontsize=9)
ax.tick_params(labelsize=8)

ax.plot(misfitsCK_hom_co['m_mis'], misfitsCK_hom_co['d_mis'],
        'o-', color='k', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"H")
ax.plot(misfitsCK_hom_co.iloc[idxCK_hom_co]['m_mis'], misfitsCK_hom_co.iloc[idxCK_hom_co]['d_mis'],
        'o', color='k', markerfacecolor='k', markersize=5,
        label=r"H preferred $\gamma$={:.1e}".format(misfitsCK_hom_co['gamma'].iloc[idxCK_hom_co]))

ax.plot(misfitsCK_sw_co['m_mis'], misfitsCK_sw_co['d_mis'],
        's-', color='red', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"S")
ax.plot(misfitsCK_sw_co.iloc[idxCK_sw_co]['m_mis'], misfitsCK_sw_co.iloc[idxCK_sw_co]['d_mis'],
        's', color='red', markerfacecolor='red', markersize=5,
        label=r"S preferred $\gamma$={:.1e}".format(misfitsCK_sw_co['gamma'].iloc[idxCK_sw_co]))

ax.plot(misfitsCK_3d_ori_co['m_mis'], misfitsCK_3d_ori_co['d_mis'],
        'D-', color='dodgerblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Original 3D")
ax.plot(misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['m_mis'], misfitsCK_3d_ori_co.iloc[idxCK_3d_ori_co]['d_mis'],
        'D', color='dodgerblue', markerfacecolor='dodgerblue', markersize=5,
        label=r"Original 3D preferred $\gamma$={:.1e}".format(misfitsCK_3d_ori_co['gamma'].iloc[idxCK_3d_ori_co]))

ax.plot(misfitsCK_3d_co['m_mis'], misfitsCK_3d_co['d_mis'],
        'D-', color='darkblue', markerfacecolor=color_L[2], linewidth=1.0, markersize=4, label=r"Amplified 3D x2.5")
ax.plot(misfitsCK_3d_co.iloc[idxCK_3d_co]['m_mis'], misfitsCK_3d_co.iloc[idxCK_3d_co]['d_mis'],
        'D', color='darkblue', markerfacecolor='darkblue', markersize=5,
        label=r"Amplified 3D x2.5 preferred $\gamma$={:.1e}".format(misfitsCK_3d_co['gamma'].iloc[idxCK_3d_co]))

ax.text(0.02, 0.02, panel_labels[1], transform=ax.transAxes, fontsize=12, fontweight="bold", fontname="Nimbus Sans",
        color='k', ha='left', va='bottom')
ax.grid(True, which='major', color='#888888', linestyle='-', alpha=0.6 )
ax.grid(True, which='minor', color='#999999', linestyle='-', alpha=0.2 )
ax.set_box_aspect(1)
ax.legend(fontsize=7)

figpath = datadir + 'figures/locking/'
output_file = figpath + f'Lcurvelock_coseis_{meshnameCK}_temp.pdf'
print(output_file)
plt.savefig(output_file, dpi=300, bbox_inches='tight')
